# Experiment with Azure Machine Learning

## Introduction

## Preprocess data and configure featurization

Create a MLTable data asset when your data is stored in a folder together with a MLTable file.

``` python
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import Input

my_training_data_input = Input(type=AssetTypes.MLTABLE, path="azureml:input-data-automl:1")
```

<div class="alert alert-info">
<b>💡Tip:</b> 

[How to create a MLTable data asset in Azure Machine Learning?](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-mltable)
</div>

## Run an automated machine learning experiment

<div class="alert alert-info">
<b>💡Tip:</b> 

[Full list of supported algorithms.](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-configure-auto-train#supported-algorithms?azure-portal=true)
</div>

### Configure an AutoML experiment

When you use the Python SDK (v2) to configure an AutoML experiment or job, you configure the experiment using the `automl` class. For classification, you use the `automl.classification` function.

``` python
from azure.ai.ml import automl

# configure the classification job
classification_job = automl.classification(
    compute="aml-cluster",
    experiment_name="auto-ml-class-dev",
    training_data=my_training_data_input,
    target_column_name="Diabetic",
    primary_metric="accuracy",
    n_cross_validations=5,
    enable_model_explainability=True
)

type(classification_job)
# azure.ai.ml.entities._job.automl.tabular.classification_job.ClassificationJob
```

<div class="alert alert-info">
<b>ℹ️Note:</b> 

AutoML needs a MLTable data asset as input. In the example, `my_training_data_input` refers to a MLTable data asset created in the Azure Machine Learning workspace.
</div>

### Specify the primary metric

One of the most important settings you must specify is the primary_metric. AutoML uses the primary metric to rank all trained models and select the best one. Azure Machine Learning supports a set of named metrics for each task type.

<div class="alert alert-info">
<b>💡Tip:</b> 

Find a full list of primary metrics and their definitions in [evaluate automated machine learning experiment results](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-understand-automated-ml).
</div>

In [5]:
from azure.ai.ml.automl import ClassificationPrimaryMetrics
 
list(ClassificationPrimaryMetrics)

[<ClassificationPrimaryMetrics.AUC_WEIGHTED: 'AUCWeighted'>,
 <ClassificationPrimaryMetrics.ACCURACY: 'Accuracy'>,
 <ClassificationPrimaryMetrics.NORM_MACRO_RECALL: 'NormMacroRecall'>,
 <ClassificationPrimaryMetrics.AVERAGE_PRECISION_SCORE_WEIGHTED: 'AveragePrecisionScoreWeighted'>,
 <ClassificationPrimaryMetrics.PRECISION_SCORE_WEIGHTED: 'PrecisionScoreWeighted'>]

In [6]:
from azure.ai.ml.automl import RegressionPrimaryMetrics
 
list(RegressionPrimaryMetrics)

[<RegressionPrimaryMetrics.SPEARMAN_CORRELATION: 'SpearmanCorrelation'>,
 <RegressionPrimaryMetrics.NORMALIZED_ROOT_MEAN_SQUARED_ERROR: 'NormalizedRootMeanSquaredError'>,
 <RegressionPrimaryMetrics.R2_SCORE: 'R2Score'>,
 <RegressionPrimaryMetrics.NORMALIZED_MEAN_ABSOLUTE_ERROR: 'NormalizedMeanAbsoluteError'>]

### Set the limits

Each model AutoML trains consumes compute resources. To control costs and training time, you can set limits on an AutoML job using `set_limits()`.

There are several options to set limits to an AutoML experiment:
- `timeout_minutes`: Number of minutes after which the complete AutoML experiment is terminated.
- `trial_timeout_minutes`: Maximum number of minutes one trial can take.
- `max_trials`: Maximum number of trials, or models that are trained.
- `enable_early_termination`: Whether to end the experiment if the score isn't improving in the short term.

``` python
classification_job.set_limits(
    timeout_minutes=60, 
    trial_timeout_minutes=20, 
    max_trials=5,
    enable_early_termination=True,
)
```

### Submit an AutoML experiment


``` python
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

# create ML client from local Azure ML config (config.json)
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

# submit the AutoML job
returned_job = ml_client.jobs.create_or_update(
    classification_job
)

# get a direct link to the AutoML job
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)
```

## Evaluate and compare models

<div class="alert alert-info">
<b>ℹ️Note:</b> 

- [model interpretability](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-machine-learning-interpretability)
- [evaluate AutoML runs](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-understand-automated-ml)
</div>

## Configure MLflow for model tracking in notebooks

### Configure MLflow for model tracking in notebooks

Check if `mlflow` and `azureml-mlflow` is installed or not.

``` bash
uv tree --package mlflow
uv tree --package azureml-mlflow

# or
uv pip show mlflow
uv pip show azureml-mlflow
```

If not installed, install the packages.

``` bash
uv add mlflow azureml-mlflow
```

Set the MLflow tracking url.

``` python
mlflow.set_tracking_uri = "<MLFLOW-TRACKING-URI>"
```

<div class="alert alert-info">
<b>💡Tip:</b> 

[Set up the tracking environment when working on a local device.](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-use-mlflow-cli-runs)
</div>


In [3]:
import mlflow
import azureml.mlflow
print(f"mlflow version:         {mlflow.__version__}")
print(f"azureml.mlflow version: {azureml.mlflow.__version__}")

mlflow version:         3.13.0
azureml.mlflow version: 1.62.0.post5


## Train and track models in notebooks

### Create a MLflow experiment

You can create a MLflow experiment, which allows you to group runs. If you don't create an experiment, MLflow assumes the default experiment with name `Default`.

``` python
import mlflow

mlflow.set_experiment(experiment_name="heart-condition-classifier")
```

### Log results with MLflow

To start a run tracked by MLflow, you use `start_run()`. Next, to track the model, you can:
- Enable autologging
- Use custom logging

#### Enable autologging

MLflow supports automatic logging for popular machine learning libraries. When enabled autologging, MLflow instructs framework to log metrics, parameters, artifacts, and models automatically. You don't need to specify what to log as the framework decides what's relevant.

You can turn on autologging by calling `mlflow.autolog()` before your training code. You can also use the framework-specific method, such as `mlflow.xgboost.autolog()`, for more granular control.

``` python
from xgboost import XGBClassifier

with mlflow.start_run():
    mlflow.autolog()

    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
```

<div class="alert alert-info">
<b>💡Tip:</b> 

List of [all supported frameworks for autologging in the official MLflow documentation](https://mlflow.org/docs/latest/ml/tracking/#automatic-logging?azure-portal=true).
</div>

#### Use custom logging

You can manually log your model with MLflow. Manually logging models is helpful, when you want to log supplementary or custom information that isn't logged through autologging.

<div class="alert alert-info">
<b>ℹ️Note:</b> 

You can choose to only use custom logging, or use custom logging in combination with autologging.
</div>

Common functions used with custom logging are:
- `mlflow.log_param()`: Logs a single key-value parameter. Use this function for an input parameter you want to log.
- `mlflow.log_metric()`: Logs a single key-value metric. Value must be a number. Use this function for any output you want to store with the run.
- `mlflow.log_figure()`: Logs a matplotlib figure directly as an artifact.
- `mlflow.log_image()`: Logs a numpy or PIL image as an artifact.
- `mlflow.log_artifact():` Logs any existing file as an artifact.
- `mlflow.log_model()`: Logs a model. Use this function to create a MLflow model, which can include a custom signature, environment, and input examples.

<div class="alert alert-info">
<b>💡Tip:</b> 

- [Official MLflow documentation](https://mlflow.org/docs/latest/ml/tracking/)
- [Azure Machine Learning documentation](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-log-view-metrics)
</div>

To use custom logging in a notebook, start a run and log any metric you want:

``` python
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

with mlflow.start_run():
    model = XGBClassifier(eval_metric="logloss")
    model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
    y_pred = model.predict(X_test)

    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)
```

## Evaluate models with the Responsible AI dashboard

### Why responsible AI matters

Models are often used when making consequential decisions. Whatever your model predicts, you should consider Microsoft's six Responsible AI principles:
- **Fairness**: Ensure your model provides equitable outcomes by testing for and mitigating harmful bias across groups.
- **Reliability & Safety**: Build, test, and monitor your model so it performs consistently and prevents unsafe behavior.
- **Privacy & Security**: Protect user data through minimal collection and responsible data-handling practices.
- **Inclusiveness**: Design and evaluate systems so people of diverse abilities and backgrounds can use them effectively.
- **Transparency**: Communicate clearly how your model works and how its outputs should be interpreted.
- **Accountability**: Assign human oversight so decisions influenced by AI remain traceable and governed.

![Responsible AI](../../images/operationalize-ml-models-mlops/responsible-ai.png)

### Create a Responsible AI dashboard

To generate a Responsible AI (RAI) dashboard, you create a pipeline using Azure Machine Learning's built-in RAI components. The pipeline must:
1. Start with the `RAI Insights dashboard constructor`.
2. Include one or more RAI tool components for the insights you need.
3. End with `Gather RAI Insights dashboard` to collect everything into one dashboard.

The available RAI tool components are:
- `Add Explanation to RAI Insights dashboard`: Shows how much each feature influences the model's predictions.
- `Add Error Analysis to RAI Insights dashboard`: Identifies subgroups of data where the model makes more errors.
- `Add Counterfactuals to RAI Insights dashboard`: Explores how changes in input would change the model's output.
- `Add Causal to RAI Insights dashboard`: Uses historical data to estimate the causal effect of features on outcomes.

You can build this pipeline using the Python SDK, the CLI, or the no-code experience in Azure Machine Learning studio.

<div class="alert alert-info">
<b>💡Tip:</b> 

[Responsible AI dashboard in Azure Machine Learning](https://learn.microsoft.com/en-us/azure/machine-learning/how-to-responsible-ai-dashboard)
</div>

## Summary

In this module, you've learned how to find the best machine learning model with Azure Machine Learning using two complementary approaches.

You've learned how to:
- Prepare your data to use AutoML for classification.
- Configure and run an AutoML experiment.
- Evaluate and compare AutoML models.
- Configure MLflow for model tracking in notebooks.
- Use MLflow for model tracking in notebooks.
- Evaluate a trained model using the Responsible AI dashboard.

<div class="alert alert-info">
<b>Note:</b> Informational message.
</div>

<div class="alert alert-success">
<b>Tip:</b> Success or tip message.
</div>

<div class="alert alert-warning">
<b>Warning:</b> Warning message.
</div>

<div class="alert alert-danger">
<b>Caution:</b> Danger or error message.
</div>

<img src="../../images/operationalize-ml-models-mlops/responsible-ai.png" alt="Responsible AI" width="1200px">